# Malware Detection Pipeline

This notebook consolidates the entire malware detection workflow, including:
1. Data Loading & Shuffling (Training and Test sets)
2. Feature Preprocessing (Scaling)
3. Heuristic Rule (8+ Zero Features)
4. Deep Learning (DNN) Training with Performance Plots
5. Baseline ML Model Training
6. Holistic Evaluation on the Shuffled APT1 dataset with Confusion Matrices and ROC Curves

In [ ]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score, roc_curve, auc
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

# --- CONFIGURATION ---
TRAIN_CSV = r'd:\btp_work\btp2_malware_features.csv'
SHUFFLED_TRAIN_CSV = r'd:\btp_work\btp2_malware_features_shuffled.csv'
TEST_CSV = r'd:\btp_work\APT1_features_btp2.csv'
SHUFFLED_TEST_CSV = r'd:\btp_work\APT1_features_shuffled.csv'
MODEL_SAVE_PATH = 'best_context_dnn.h5'
SCALER_SAVE_PATH = 'scaler_context.pkl'
GRAPH_DIR = 'model_graphs'
RANDOM_SEED = 42
ZERO_FEATURE_THRESHOLD = 8

if not os.path.exists(GRAPH_DIR):
    os.makedirs(GRAPH_DIR)

CONTEXT_COLS = [
    'FileSize','Entropy','Imports_Count','Machine','SizeOfOptionalHeader',
    'Characteristics','NumberOfSections','TimeDateStamp','MajorLinkerVersion',
    'SizeOfCode','SizeOfInitializedData','SizeOfUninitializedData',
    'AddressOfEntryPoint','BaseOfCode','ImageBase','SectionAlignment',
    'FileAlignment','MajorOperatingSystemVersion','SizeOfImage',
    'SizeOfHeaders','CheckSum','Subsystem','DllCharacteristics',
    'SizeOfStackReserve','SizeOfHeapCommit','NumberOfRvaAndSizes',
    'Entropy_Mean','Entropy_Max'
]

## 1. Data Loading & Shuffling

In [ ]:
print("[+] Shuffling Training Dataset...")
df_train = pd.read_csv(TRAIN_CSV)
df_train_shuffled = df_train.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
df_train_shuffled.to_csv(SHUFFLED_TRAIN_CSV, index=False)
print(f"[+] Training set shuffled and saved to {SHUFFLED_TRAIN_CSV}")

print("[+] Shuffling Test Dataset (APT1)...")
df_test = pd.read_csv(TEST_CSV)
df_test_shuffled = df_test.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
df_test_shuffled.to_csv(SHUFFLED_TEST_CSV, index=False)
print(f"[+] Test set shuffled and saved to {SHUFFLED_TEST_CSV}")

# Prepare features for training
if 'filename' in df_train_shuffled.columns:
    df_train_shuffled = df_train_shuffled.drop(columns=['filename'])

all_valid_features = [c for c in CONTEXT_COLS if c in df_train_shuffled.columns]
X_train_full = df_train_shuffled[all_valid_features].fillna(0)
y_train_full = df_train_shuffled['Label'].values

print(f"[+] Training with {len(all_valid_features)} features.")

## 2. Preprocessing & Heuristic Rule

In [ ]:
def apply_zero_feature_heuristic(X_df, feature_names):
    """Flags samples with more than 8 zeroed features as malware directly."""
    if isinstance(X_df, np.ndarray):
        X_df = pd.DataFrame(X_df, columns=feature_names)
    zero_counts = (X_df == 0).sum(axis=1)
    return (zero_counts >= ZERO_FEATURE_THRESHOLD).astype(int).values

# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train_full)
joblib.dump(scaler, SCALER_SAVE_PATH)
print(f"[+] Scaler saved to {SCALER_SAVE_PATH}")

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_train_full, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train_full
)

## 3. Deep Learning Model (DNN) Training

In [ ]:
def build_dnn_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
    return model

dnn_model = build_dnn_model(X_train.shape[1])
early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("[+] Training DNN Model...")
history = dnn_model.fit(
    X_train, y_train, 
    validation_data=(X_val, y_val), 
    epochs=50, batch_size=32, callbacks=[early_stopping], verbose=1
)
dnn_model.save(MODEL_SAVE_PATH)

# Plot Training History
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('DNN Accuracy')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('DNN Loss')
plt.legend()
plt.savefig(os.path.join(GRAPH_DIR, 'dnn_training_history.png'))
plt.show()

## 4. Baseline ML Models Training

In [ ]:
ml_models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'SVM': SVC(probability=True, random_state=RANDOM_SEED),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_SEED),
    'MLP (Sklearn)': MLPClassifier(max_iter=500, random_state=RANDOM_SEED)
}

for name, model in ml_models.items():
    print(f"[+] Training {name}...")
    model.fit(X_train, y_train)

## 5. Evaluation on Shuffled APT1 Dataset

In [ ]:
print("\n[+] Evaluating on Shuffled APT1 Dataset...")
df_apt1 = pd.read_csv(SHUFFLED_TEST_CSV)
y_test_apt1 = df_apt1['Label'].values
X_test_apt1_raw = df_apt1[all_valid_features].fillna(0)
X_test_apt1_scaled = scaler.transform(X_test_apt1_raw)

heuristic_pred = apply_zero_feature_heuristic(X_test_apt1_raw, all_valid_features)
results_data = []

def plot_confusion_matrix(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix: {name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(os.path.join(GRAPH_DIR, f'confusion_matrix_{name.replace(" ", "_").lower()}.png'))
    plt.show()

plt.figure(figsize=(10, 8))

# DNN Evaluation
y_prob_dnn = dnn_model.predict(X_test_apt1_scaled).flatten()
y_pred_dnn = (y_prob_dnn > 0.5).astype(int)
final_pred_dnn = np.maximum(y_pred_dnn, heuristic_pred)
plot_confusion_matrix(y_test_apt1, final_pred_dnn, 'DNN (Context)')

results_data.append({
    'Model': 'DNN (Context)',
    'Accuracy': accuracy_score(y_test_apt1, final_pred_dnn),
    'F1-Score': f1_score(y_test_apt1, final_pred_dnn)
})

fpr, tpr, _ = roc_curve(y_test_apt1, y_prob_dnn)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'DNN (AUC = {roc_auc:.2f})')

# ML Models Evaluation
for name, model in ml_models.items():
    y_prob = model.predict_proba(X_test_apt1_scaled)[:, 1]
    y_pred = model.predict(X_test_apt1_scaled)
    final_pred = np.maximum(y_pred, heuristic_pred)
    
    plot_confusion_matrix(y_test_apt1, final_pred, name)
    
    results_data.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_apt1, final_pred),
        'F1-Score': f1_score(y_test_apt1, final_pred)
    })
    
    fpr, tpr, _ = roc_curve(y_test_apt1, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison (on APT1)')
plt.legend(loc="lower right")
plt.savefig(os.path.join(GRAPH_DIR, 'roc_comparison.png'))
plt.show()

df_final_res = pd.DataFrame(results_data)
print("\n--- Final APT1 Results (Shuffled Data) ---")
print(df_final_res.sort_values('Accuracy', ascending=False))
df_final_res.to_csv('apt1_evaluation_results.csv', index=False)

plt.figure(figsize=(10, 5))
sns.barplot(x='Accuracy', y='Model', data=df_final_res.sort_values('Accuracy', ascending=False))
plt.title('Model Accuracy Comparison')
plt.show()